# NBA player archetypes — final 2025–26 clustering

This notebook is the complete, runnable record of the final seven-archetype model. Run cells in order from the repository root or from `notebooks/`. It reads the saved 389-player modeling table and position reference, then writes assignments, relative fit scores, and model metadata to `data/processed/`.

**The results are determined by the saved input data and the exact settings below.** Position is retained for display; it does not choose a cluster. LEBRON is used later for website rankings, not for clustering. After running this notebook, `python web/build_data.py` refreshes the website data.

## 1. Setup and feature design

Each feature is capped at its pool's 1st and 99th percentiles, standardized within that pool, then weighted by feature family. Dividing each family's weight by the square root of its feature count keeps a family with many columns from dominating simply because it has more columns. The high-usage and role-player pools have separate feature sets and scalers.

The constants, feature lists, weights, random seeds, and helper functions below are the final model settings.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/processed/modeling_players.csv").exists(), "Run this notebook from the repo root or notebooks/"
PROCESSED = ROOT / "data" / "processed"
SEED = 2
USAGE_THRESHOLD = 25.0

ROLE_GROUPS = {
    "secondary_scoring_volume": ["fga_per_36"],
    "shot_profile": ["three_rate", "fta_rate", "rim_rate", "short_mid_rate"],
    "rebounding": ["oreb_per_36", "dreb_per_36"],
    "defensive_activity": ["stl_per_36", "blk_per_36", "dws_per_36", "dbpm"],
    "catch_and_shoot": ["catch_shoot_fga_per_36", "catch_shoot_fga_share"],
    "secondary_creation": ["ast_per_36", "tov_per_36"],
    "passing_context": ["passing_passes_made_per_36", "passing_potential_ast_per_36", "passing_ast_adj_per_36", "passing_ast_pts_created_per_36"],
}
# Fans tend to distinguish players more by their shooting and defensive role
# than by ordinary defensive-rebound totals. The other original weights stay put.
ROLE_WEIGHTS = {
    "secondary_scoring_volume": 1.10,
    "shot_profile": 0.90,
    "rebounding": 0.80,
    "defensive_activity": 1.40,
    "catch_and_shoot": 0.85,
    "secondary_creation": 1.05,
    "passing_context": 0.70,
}
HIGH_GROUPS = {
    "scoring_volume": ["fga_per_36", "fta_rate"],
    "shot_location": ["three_rate", "rim_rate", "short_mid_rate", "long_mid_rate"],
    "playmaking": ["ast_per_36", "ast_to_tov_ratio"],
    "passing_context": ROLE_GROUPS["passing_context"],
    "frontcourt_involvement": ["oreb_per_36", "dreb_per_36", "blk_per_36"],
}
# Rebounding and blocks provide positional context without defining a star's role.
HIGH_WEIGHTS = {"scoring_volume": 1.25, "shot_location": 1.0, "playmaking": 1.30, "passing_context": 0.70, "frontcourt_involvement": 0.90}


def weighted_matrix(frame: pd.DataFrame, groups: dict, weights: dict, reference: pd.DataFrame | None = None) -> np.ndarray:
    features = [feature for members in groups.values() for feature in members]
    reference = frame if reference is None else reference
    capped_reference = reference[features].copy()
    capped = frame[features].copy()
    if capped.isna().any().any() or capped_reference.isna().any().any():
        raise ValueError("Missing model feature values")
    for feature in features:
        low, high = capped_reference[feature].quantile([0.01, 0.99])
        capped_reference[feature] = capped_reference[feature].clip(low, high)
        capped[feature] = capped[feature].clip(low, high)
    matrix = StandardScaler().fit(capped_reference).transform(capped)
    for group, members in groups.items():
        indices = [features.index(member) for member in members]
        matrix[:, indices] *= weights[group] / np.sqrt(len(members))
    return matrix


def relative_fit(matrix: np.ndarray, centers: np.ndarray) -> np.ndarray:
    """Inverse-square distance shares; descriptive fit, not probabilities."""
    distances = np.linalg.norm(matrix[:, None, :] - centers[None, :, :], axis=2)
    inverse_square = 1.0 / np.maximum(distances, 1e-9) ** 2
    return 100.0 * inverse_square / inverse_square.sum(axis=1, keepdims=True)


## 2. Load the saved player table

The 500-minute eligibility filter and source-data preparation are already reflected in `modeling_players.csv`. This notebook checks that all 389 eligible players have a listed position, which is carried through to the output for reference.

In [2]:
players = pd.read_csv(PROCESSED / "modeling_players.csv")
positions = pd.read_csv(ROOT / "data/raw/nba_2025_26_player_positions.csv")
players = players.merge(positions, on="Player", how="left", validate="one_to_one")
assert len(players) == 389 and players.listed_position.notna().all()

players[["Player", "usage_pct", "listed_position"]].head()


          Player  usage_pct listed_position
0     A.J. Green       14.0              SG
1     AJ Johnson       20.6              SG
2   Aaron Gordon       21.9              PF
3  Aaron Holiday       16.6              PG
4  Aaron Nesmith       19.8              SF

## 3. Cluster high-usage players

Players at **25% usage or above** enter one two-cluster K-means model. The playmaking family uses assists per 36 and assist-to-turnover ratio. The cluster with the higher mean assists per 36 is named **Primary Playmakers**; the other is **Primary Scorers**. No position rule or player-level exception changes an assignment.

In [3]:
# Fit every high-usage player together. Position does not determine a group.
high = players[players.usage_pct >= USAGE_THRESHOLD].copy()
if high.tov_per_36.le(0).any():
    raise ValueError("Assist-to-turnover ratio requires positive turnovers per 36")
high["ast_to_tov_ratio"] = high.ast_per_36 / high.tov_per_36
high_matrix = weighted_matrix(high, HIGH_GROUPS, HIGH_WEIGHTS)
high_model = KMeans(n_clusters=2, random_state=7, n_init=50).fit(high_matrix)
high["cluster"] = high_model.labels_
high_assists = high.groupby("cluster").ast_per_36.mean()
playmaker_cluster = int(high_assists.idxmax())
high_labels = {playmaker_cluster: "Primary Playmakers", 1 - playmaker_cluster: "Primary Scorers"}
high["archetype"] = high.cluster.map(high_labels)
high_order = ["Primary Playmakers", "Primary Scorers"]
high_centers = np.vstack([high_model.cluster_centers_[cluster] for label in high_order for cluster, name in high_labels.items() if name == label])
high_fit = pd.DataFrame(relative_fit(high_matrix, high_centers), columns=high_order, index=high.index)

high.groupby("archetype").agg(players=("Player", "size"), mean_assists_per_36=("ast_per_36", "mean"))


                    players  mean_assists_per_36
archetype                                       
Primary Playmakers       26             7.757490
Primary Scorers          39             4.019004

## 4. Select the role-player cluster count

Players below 25% usage enter one K-means fit over the full role-player pool. Evaluate **K = 5 through 12**, requiring every cluster to contain at least 4% of role players. Of eligible candidates, choose the highest silhouette score (breaking ties toward smaller K). The assertion makes a changed selection explicit because the five names below would then need review.

In [4]:
role = players[players.usage_pct < USAGE_THRESHOLD].copy().reset_index(drop=True)
matrix = weighted_matrix(role, ROLE_GROUPS, ROLE_WEIGHTS)
selection = []
models = {}
for k in range(5, 13):
    model = KMeans(n_clusters=k, random_state=SEED, n_init=50).fit(matrix)
    counts = np.bincount(model.labels_)
    selection.append({"k": k, "silhouette": float(silhouette_score(matrix, model.labels_)), "smallest_cluster": int(counts.min()), "smallest_share": float(counts.min() / len(role))})
    models[k] = model
eligible = [row for row in selection if row["smallest_share"] >= 0.04]
selected = max(eligible or selection, key=lambda row: (row["silhouette"], -row["k"]))
model = models[selected["k"]]
assert selected["k"] == 5, "Review role labels after K selection changes"
role["cluster"] = model.labels_

pd.DataFrame(selection).assign(eligible=lambda rows: rows.smallest_share >= 0.04)


    k  silhouette  smallest_cluster  smallest_share  eligible
0   5    0.151126                48        0.148148      True
1   6    0.150770                46        0.141975      True
2   7    0.149731                23        0.070988      True
3   8    0.144542                22        0.067901      True
4   9    0.147882                15        0.046296      True
5  10    0.142629                15        0.046296      True
6  11    0.141863                14        0.043210      True
7  12    0.132515                14        0.043210      True

## 5. Name the role-player clusters and calculate fit

Names follow aggregate cluster traits in a fixed order: highest rim rate, then assists, then shot volume, then defensive box plus/minus among the remaining clusters. The final cluster is **Spot-Up Shooters**. The three-point-rate assertion checks the interpretation of **3-and-D Contributors**.

Fit percentages are inverse-square distance shares across centroids **within the same pool**. They add to 100% per player but are descriptive similarity scores, not calibrated probabilities. High-usage and role-player fit values should not be compared with one another.

In [5]:
# Name the five clusters from their aggregate profiles, never from an
# individual player's desired assignment or a second clustering stage.
profile = role.groupby("cluster")[["rim_rate", "ast_per_36", "fga_per_36", "dbpm", "three_rate"]].mean()
remaining_clusters = set(profile.index)
role_labels = {}
for feature, label in [("rim_rate", "Rim-Runners"), ("ast_per_36", "Secondary Facilitators"), ("fga_per_36", "Secondary Scorers"), ("dbpm", "3-and-D Contributors")]:
    cluster = profile.loc[sorted(remaining_clusters), feature].idxmax()
    role_labels[int(cluster)] = label
    remaining_clusters.remove(cluster)
role_labels[int(remaining_clusters.pop())] = "Spot-Up Shooters"
assert profile.loc[[c for c, label in role_labels.items() if label == "3-and-D Contributors"], "three_rate"].iloc[0] > 0.45
role["archetype"] = role.cluster.map(role_labels)
role_order = ["3-and-D Contributors", "Rim-Runners", "Secondary Facilitators", "Secondary Scorers", "Spot-Up Shooters"]
role_centers = np.vstack([model.cluster_centers_[next(cluster for cluster, name in role_labels.items() if name == label)] for label in role_order])
role_fit = pd.DataFrame(relative_fit(matrix, role_centers), columns=role_order, index=role.index)

role.groupby("archetype").agg(players=("Player", "size"), mean_three_rate=("three_rate", "mean"), mean_assists_per_36=("ast_per_36", "mean")).sort_index()


                        players  mean_three_rate  mean_assists_per_36
archetype                                                            
3-and-D Contributors         63         0.553784             3.070155
Rim-Runners                  48         0.083653             2.411569
Secondary Facilitators       60         0.396629             6.485812
Secondary Scorers            65         0.443358             3.493122
Spot-Up Shooters             88         0.546681             2.483905

## 6. Check stability

Refit the selected role-player K-means model on twenty 80% subsamples. The mean adjusted Rand index compares each subsample's labels with the final fit on those same players. This is a diagnostic; it does not alter any assignment.

In [6]:
rng = np.random.default_rng(SEED)
stability = []
for _ in range(20):
    index = np.sort(rng.choice(len(role), size=round(0.8 * len(role)), replace=False))
    sample_labels = KMeans(n_clusters=selected["k"], random_state=int(rng.integers(0, 2**31 - 1)), n_init=25).fit_predict(matrix[index])
    stability.append(adjusted_rand_score(model.labels_[index], sample_labels))

float(np.mean(stability))


0.7103576224137471

## 7. Save the final results

Write the same three artifacts consumed by the rest of the project:

- `current_player_archetypes.csv`: one assigned archetype per player;
- `player_cluster_fit.csv`: within-pool centroid-distance shares and the assigned share;
- `current_archetype_metadata.json`: weights, K selection, diagnostics, counts, and cluster means.

The outputs are deterministic for the saved input and these settings.

In [7]:
output = pd.concat([high, role], ignore_index=True)
output["tier"] = np.where(output.usage_pct >= USAGE_THRESHOLD, "high", "role")
assert output.Player.is_unique and len(output) == 389
output[["Player", "listed_position", "tier", "usage_pct", "cluster", "archetype"]].to_csv(PROCESSED / "current_player_archetypes.csv", index=False)
fit_table = pd.concat([
    high[["Player", "listed_position", "archetype"]].assign(tier="high").join(high_fit),
    role[["Player", "listed_position", "archetype"]].assign(tier="role").join(role_fit),
], ignore_index=True)
fit_table[high_order + role_order] = fit_table[high_order + role_order].round(2)
fit_table["assigned_fit_pct"] = fit_table.apply(lambda row: row[row["archetype"]], axis=1)
fit_table = fit_table.sort_values("Player").reset_index(drop=True)
fit_table.to_csv(PROCESSED / "player_cluster_fit.csv", index=False)
profile_features = ["three_rate", "catch_shoot_fga_share", "catch_shoot_fga_per_36", "stl_per_36", "blk_per_36", "dbpm", "oreb_per_36", "dreb_per_36", "ast_per_36", "fga_per_36", "rim_rate"]
role_profiles = role.groupby("archetype")[profile_features].mean().round(3).to_dict(orient="index")
metadata = {
    "source": "data/processed/modeling_players.csv",
    "method": "Single K-means fit over the complete role-player pool; silhouette selects K among models with no group below 4%",
    "usage_threshold": USAGE_THRESHOLD,
    "role_weights": ROLE_WEIGHTS,
    "high_usage_weights": HIGH_WEIGHTS,
    "high_usage_features": HIGH_GROUPS,
    "high_usage_method": "Two K-means clusters over all high-usage players; names follow cluster mean assists per 36",
    "high_usage_silhouette": float(silhouette_score(high_matrix, high_model.labels_)),
    "selection": selection,
    "selected_k": selected["k"],
    "silhouette": selected["silhouette"],
    "bootstrap_stability_ari": float(np.mean(stability)),
    "counts": output.archetype.value_counts().sort_index().to_dict(),
    "role_labels": role_labels,
    "role_cluster_means": role_profiles,
}
(PROCESSED / "current_archetype_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Selected {selected['k']} role clusters; silhouette {selected['silhouette']:.3f}; bootstrap ARI {metadata['bootstrap_stability_ari']:.3f}")
print(output.archetype.value_counts().sort_index().to_string())


Selected 5 role clusters; silhouette 0.151; bootstrap ARI 0.710
archetype
3-and-D Contributors      63
Primary Playmakers        26
Primary Scorers           39
Rim-Runners               48
Secondary Facilitators    60
Secondary Scorers         65
Spot-Up Shooters          88
